# Dōkō (同行) — Walking Together Through Scripture

**An AI Scripture companion for Mandarin and Japanese language learners.**

*Dōkō* (同行) means "walking together."

---

My grandpa spoke only Mandarin. I learned Mandarin so I could stay close to him — often
with a translation app open in the middle of our conversations. He passed away last year.

Dōkō carries that forward for my church's Mandarin-speaking community: it helps someone
read Scripture in a language they are still learning, and it never puts a machine
translation of the Bible in front of them to do it.

- **Repository:** https://github.com/jamesianfoo/doko_youversion
- **APIs used:** YouVersion Platform API (Scripture) + Gloo AI Studio (explanation)

## The design rule this project is built around

The tempting shortcut in a project like this is to let an LLM translate or paraphrase
Scripture. Dōkō never does. AI is used for *explanation only* — never for source text,
and never for factual reference data.

| Layer | Source | AI involved? |
|---|---|---|
| Bible text | YouVersion Platform API — real licensed translations | **No** |
| Pinyin / furigana | `pypinyin`, `pykakasi` — deterministic libraries | **No** |
| Greek words, Strong's numbers, lexical definitions | Bundled public-domain lexicon data | **No** |
| Word explanations | Gloo AI Studio | Yes |
| "Behind the translation" insight | Gloo AI Studio, *grounded in the real Greek above* | Yes — prose only |

The last row is the interesting one, and the final section of this notebook
**verifies it programmatically**: every Greek word the model cites is checked against
the real lexicon file it was given.

This notebook imports the exact same `api_clients.py` the web app uses — the calls
demonstrated here are the calls that run in production, not a parallel reimplementation.

## Setup

> **Kaggle:** turn **Internet on** in the notebook settings sidebar — every call below
> hits a live API. Add your credentials under **Add-ons → Secrets** as
> `YVP_APP_KEY`, `GLOO_CLIENT_ID`, and `GLOO_CLIENT_SECRET`.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/jamesianfoo/doko_youversion.git"
ON_KAGGLE = os.path.exists("/kaggle/working")

if ON_KAGGLE:
    if not os.path.exists("doko_youversion"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    sys.path.insert(0, "doko_youversion")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "requests", "python-dotenv", "pypinyin", "pykakasi"],
        check=True,
    )
else:
    sys.path.insert(0, ".")

print("Running on:", "Kaggle" if ON_KAGGLE else "local checkout")

### Credentials

Secrets are never written into this notebook. On Kaggle they come from Kaggle Secrets;
locally from a gitignored `.env` (see `.env.example` for the shape).

This runs **before** `import api_clients`, because that module reads the environment
at import time.

In [ ]:
REQUIRED = ("YVP_APP_KEY", "GLOO_CLIENT_ID", "GLOO_CLIENT_SECRET")


def load_credentials():
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for key in REQUIRED:
            os.environ[key] = secrets.get_secret(key)
        return "Kaggle Secrets"
    except Exception:
        from dotenv import load_dotenv
        load_dotenv()
        return ".env file"


source = load_credentials()
missing = [k for k in REQUIRED if not os.environ.get(k)]

print(f"Credentials loaded from: {source}")
if missing:
    print("MISSING:", ", ".join(missing), "-- the live calls below will fail.")
else:
    print("All three credentials present.")

---
# Part 1 — Real Scripture, from YouVersion

Two things matter here, and both are easy to get quietly wrong.

**First: licensed is not the same as discoverable.** YouVersion's catalog can be queried
with `all_available=true`, which returns many versions this app key cannot actually fetch
passages from — they return `403` at read time. `list_versions()` deliberately omits that
flag, so it only returns versions real enough to display.

**Second: the well-known translations are not always available.** For this app key, CUV
(Chinese Union Version) and NIV are not licensed. Rather than substitute an AI translation,
Dōkō uses other genuinely licensed published translations: **CSBS** for Chinese and
**BSB** for English.

In [ ]:
import api_clients

chinese_versions = api_clients.list_versions("zho")

print(f"{len(chinese_versions)} Chinese versions licensed for this key:\n")
for v in chinese_versions[:8]:
    print(f"  {v['id']:>6}  {v['abbreviation']:<10} {v['title']}")

In [ ]:
meta = api_clients.get_version_meta(api_clients.CHINESE_VERSION_ID)

print("Version used as the Chinese primary text:")
for key, value in meta.items():
    print(f"  {key:<14} {value}")

### Fetching a chapter with its real structure

Dōkō requests `format=html` rather than `format=text`. The HTML carries the publisher's
own paragraph breaks (`div.p`) and verse markers (`yv-v`), which lets the reader render
flowing paragraphs exactly like a real Bible app — instead of a stack of one-box-per-verse
cards. `get_chapter()` parses that into paragraphs of verses in a single API call.

In [ ]:
chapter = api_clients.get_chapter(
    api_clients.CHINESE_VERSION_ID, book="EPH", chapter=2
)

print("Reference:", chapter["reference"])
print("Paragraphs:", len(chapter["paragraphs"]))
print("Verses per paragraph:", [len(p) for p in chapter["paragraphs"]])

# get_chapter returns verse numbers as strings, straight from the markup.
verse_8 = next(
    v for para in chapter["paragraphs"] for v in para if str(v["number"]) == "8"
)
print("\nEphesians 2:8 (CSBS, real licensed text):")
print(" ", verse_8["text"])

---
# Part 2 — The reading aid (no AI)

Pinyin and furigana come from `pypinyin` and `pykakasi` — deterministic libraries, not a
model. `annotate_reading()` dispatches on the version's language tag and returns a uniform
`[{char, pinyin}, ...]` shape so the frontend renders `<ruby>` identically for both scripts.

In [ ]:
zh_chars = api_clients.annotate_reading("因为你们得救是本乎恩", "zh")
ja_chars = api_clients.annotate_reading("神の恵みにより", "ja")

print("Chinese -> pinyin")
for c in zh_chars[:8]:
    print(f"  {c['char']}  {c['pinyin']}")

print("\nJapanese -> furigana")
for c in ja_chars[:8]:
    print(f"  {c['char']}  {c['pinyin']}")

---
# Part 3 — Gloo AI for explanation

Now the AI. Gloo AI Studio authenticates with OAuth2 client credentials, then serves an
OpenAI-compatible Responses API.

Note what is being explained: a word the reader tapped, **grounded in the verse they are
actually reading**. The explanation is written in the reader's own study language, so a
Mandarin learner gets Mandarin — immersion rather than constant translation back to English.

In [ ]:
verse_text = verse_8["text"]

explanation = api_clients.explain_word(
    word="恩典",
    verse_text=verse_text,
    language_tag="zh",
    level="native",
)

print("Word: 恩典  (grace)\n")
print(explanation)

### Adjustable depth

The same word, at a beginner reading level. This is independent of *which* language the
explanation is written in — a true beginner in Mandarin still needs simple Mandarin, not
English.

In [ ]:
beginner = api_clients.explain_word(
    word="恩典", verse_text=verse_text, language_tag="zh", level="beginner"
)

print("BEGINNER level:\n")
print(beginner)

---
# Part 4 — The original Greek, which is *not* AI-generated

A friend asked whether Dōkō could show the original Hebrew/Greek. It can — but this is
exactly the kind of data an LLM should never be trusted to produce. Models hallucinate
plausible-but-wrong Strong's numbers, roots, and transliterations, and getting that wrong
is something any reader who knows Greek would immediately catch.

So this data is **real and bundled**, derived from `tahmmee/interlinear_bibledata` (built
from the public-domain Strong's Concordance plus a KJV interlinear NT). See `data/README.md`
for full sourcing and `scripts/build_greek_data.py` for exactly how it was generated,
including a deterministic — not guessed — transliteration.

Ephesians is a New Testament letter, so this chapter is **Koine Greek**. There is no Hebrew
in this demo; the same upstream source has a Hebrew Old Testament dataset if Dōkō ever
covers an OT chapter.

In [ ]:
from greek_data import get_verse_words

words = get_verse_words(8)

print(f"Ephesians 2:8 — {len(words)} Greek words, in the original word order:\n")
for w in words:
    gloss = f'"{w["gloss"]}"' if w["gloss"] else ""
    print(f"  {w['greek']:<14} {w['translit']:<14} {w['strongs']:<7} {gloss}")

---
# Part 5 — Grounding: Gloo writes the prose, the data supplies the facts

A bare word list turned out not to be very useful. Early feedback on it was blunt: it
*"doesn't resonate my soul and no path of knowing how to apply to my life."* The Greek
earns its place when it **illuminates the verse**, not when it is dumped as reference data
the reader has to interpret alone.

So Gloo writes a short insight in this shape:

> quote a phrase → say plainly what it means → let one Greek word add what the translation
> alone cannot carry → land on how it is lived

The safety property is structural, not just prompt-deep:

1. The Greek word list is loaded **server-side** from the bundled lexicon file and passed
   into the prompt as grounding — it is never accepted from the client.
2. The prompt **forbids recalling any Greek from model memory** and restricts citations to
   the supplied list.
3. The next cell **verifies the output** against the source data.

In [ ]:
insight = api_clients.explain_verse_with_greek(
    verse_text="For it is by grace you have been saved through faith, "
               "and this not from yourselves; it is the gift of God,",
    verse_ref="Ephesians 2:8",
    greek_words=words,
    language_tag="en",
)

print(insight)

### Verifying that no Greek was hallucinated

This is the claim worth checking rather than asserting. Every Greek-script token in Gloo's
prose is extracted and matched against the real lexicon rows the model was given.

In [ ]:
import re

GREEK_SCRIPT = r"[\u0370-\u03FF\u1F00-\u1FFF]+"


def normalize(word):
    # The source data keeps the verse's own punctuation attached (e.g. "δῶρον·").
    return word.rstrip("·;,.:").strip()


cited = {normalize(t) for t in re.findall(GREEK_SCRIPT, insight)}
grounded = {normalize(w["greek"]) for w in words}

print("Greek cited by Gloo:", ", ".join(sorted(cited)) or "(none)")

unbacked = cited - grounded
if unbacked:
    print("\nFAIL - not present in the real lexicon data:", unbacked)
else:
    print(f"\nPASS - all {len(cited)} verified against data/greek_ephesians_2.json")

for word in sorted(cited):
    match = next(w for w in words if normalize(w["greek"]) == word)
    print(f"  {word:<12} -> {match['strongs']:<7} {match['translit']}")

---
## Summary

| Capability | How it works |
|---|---|
| Read a chapter in Mandarin or Japanese | Real licensed YouVersion translations, publisher's own paragraph structure |
| Reading aid | Deterministic pinyin/furigana, toggleable off as fluency grows |
| Tap any word | Gloo explains it in context, in the language being learned |
| Adjustable depth | Beginner ↔ native, independent of language |
| Compare translations | Any two licensed versions side by side |
| Behind the translation | Gloo insight **grounded in real Strong's data**, verified above |

### The line this project holds

Scripture text is never machine-translated, and factual reference data is never generated.
AI is used where it genuinely helps — explaining a word to someone still learning the
language — and kept away from everything where being confidently wrong would matter.

**Repository:** https://github.com/jamesianfoo/doko_youversion